# Generate `zenodo.json`

This notebooks generate the metadata required for Zenodo author list on every new release.

In [ ]:
import os
import logging
import subprocess
import tempfile
import json
import orcid
import pandas as pd

## Reading git commit log

In [ ]:
with tempfile.NamedTemporaryFile() as fh:
    sub_proc = subprocess.run(
        'git log --pretty=short | git shortlog -sne > {0}'.format(fh.name), shell=True)
    cur_author_log = pd.read_csv(fh.name, delimiter='\t', names=['commits', 'raw_name'])

author_log = cur_author_log.groupby('raw_name').sum().reset_index()

In [ ]:
# sort by number of commits
author_log = author_log.sort_values('commits').reset_index(drop=True)

In [ ]:
# Merge double last names
names = author_log.raw_name.str.split()
mask = names.map(lambda x: True if len(x) > 3 else False)

for i in names.loc[mask].index:
    names[i][1] = names[i][1] + ' ' + names[i][2]
    del(names[i][2])

> **NOTE:** Commits from accounts with incomplete names do not count for the final table. Update `.mailmap` in tardis-sn/tardis repo if you want to include those commits.

In [ ]:
mask = names.map(lambda x: True if len(x) < 3 else False)

# Incomplete author names
for name in names[mask]:
    print(f'Incomplete author name: {name}')

In [ ]:
# Mark incomplete names to be removed later
for i in names.loc[mask].index:
    names[i] = ['DROP ME', 'DROP ME', 'DROP ME']

In [ ]:
# remove angle brackets from email addresses and create cleaned log
cleaned_log = pd.DataFrame([(item[0], item[1], item[2].strip('<>')) for item in names], 
                  columns=['first_name', 'last_name', 'email'])

In [ ]:
# Update the author log and drop the raw names from git
author_log = author_log.join(cleaned_log)
del author_log['raw_name']

In [ ]:
# use email as the index
author_log.set_index('email', inplace=True)

## Drop authors

In [ ]:
# Drop incomplete names
author_log = author_log.drop(['DROP ME'])

> **NOTE:** In case of duplicated authors, we keep the email with the highest number of commits. Update `.mailmap` in tardis-sn/tardis repo if you want to include commits from all the email adresses.

In [ ]:
# Look for duplicated author names
mask = author_log.duplicated(subset=['first_name', 'last_name'], keep=False)
author_log[mask]

# duplicate author name warning
for author in author_log[mask].itertuples():
    print(f'Duplicated author: {author.last_name}, {author.first_name} <{author.Index}>')

In [ ]:
author_log = author_log.sort_values('commits', ascending=False).drop_duplicates(subset=['first_name', 'last_name'])

In [ ]:
author_log

In [ ]:
# Drop CI users
author_log = author_log.drop(['tardis.sn.bot@gmail.com'])

## ORCID & Ordering with data

In [ ]:
orcid_data = pd.read_csv('.orcid.csv').set_index('email')

In [ ]:
author_log['orcid'] = orcid_data['orcid']

In [ ]:
author_log

### Read ORCID information ###

In [ ]:
keys = json.load(open('key_secret.json'))
api = orcid.PublicAPI(keys['key'], keys['secret'])
token = api.get_search_token_from_orcid()

In [ ]:
zenodo_authors = []
for email, author_info in author_log.sort_values('ordering').iterrows():

    zenodo_entry = {}
    zenodo_entry['name'] = '{0}, {1}'.format(author_info.last_name, author_info.first_name)

    if not pd.isnull(author_info.orcid):
        zenodo_entry['orcid'] = author_info.orcid
        author_orcid = api.read_record_public(author_info.orcid, 'record', token)

        try:
            organisation = author_orcid['activities-summary']['employments']['employment-summary'][0]['organization']['name']

        except IndexError:
            pass

        else:
            zenodo_entry["affiliation"] = organisation
            print(author_info.last_name, organisation)

    zenodo_authors.append(zenodo_entry)

In [ ]:
zenodo_json = {'creators': zenodo_authors}

In [ ]:
json.dump(zenodo_json, open('.zenodo.json', 'w'), indent=2)

In [ ]:
!cat .zenodo.json